# Project 01: Santander 고객 데이터 분석
인공지능의 이해


## 데이터 불러오기 및 전처리

In [ ]:
import pandas as pd

sample = pd.read_csv("train_ver2.csv", nrows=5)
print(sample.shape)
sample.T

(5, 48)


,0,1,2,3,4
fecha_dato,2015-01-28,2015-01-28,2015-01-28,2015-01-28,2015-01-28
ncodpers,1375586,1050611,1050612,1050613,1050614
ind_empleado,N,N,N,N,N
pais_residencia,ES,ES,ES,ES,ES
sexo,H,V,V,H,V
age,35,23,23,22,23
fecha_alta,2015-01-12,2012-08-10,2012-08-10,2012-08-10,2012-08-10
ind_nuevo,0,0,0,0,0
antiguedad,6,35,35,35,35
indrel,1,1,1,1,1


In [ ]:
product_cols = list(sample.columns[24:48])
print(len(product_cols))
print(product_cols)

24
['ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1']


In [ ]:
info_cols = ["fecha_dato", "ncodpers", "ind_empleado", "sexo", "age",
             "tiprel_1mes", "nomprov", "renta", "segmento"]

# 글자 열은 category, 고객 ID는 int32, 상품 열은 일단 float32(빈 값 때문)
dtypes = {c: "category" for c in ["fecha_dato", "ind_empleado", "sexo", "age",
                                  "tiprel_1mes", "nomprov", "segmento"]}
dtypes["ncodpers"] = "int32"
for c in product_cols:
    dtypes[c] = "float32"

df = pd.read_csv("train_ver2.csv", usecols=info_cols + product_cols, dtype=dtypes)

# 상품 열: 빈 값은 0(미보유)으로 채우고 int8로 변환
df[product_cols] = df[product_cols].fillna(0).astype("int8")

# age: 공백 제거 후 숫자로 변환 (" NA"는 NaN이 됨)
df["age"] = pd.to_numeric(df["age"].astype(str).str.strip(), errors="coerce")

print(df.shape)

(13647309, 33)


In [ ]:
df.info(memory_usage="deep")
print(list(df["fecha_dato"].cat.categories))
print(df[["sexo", "age", "nomprov", "renta", "segmento"]].isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 13647309 entries, 0 to 13647308
Data columns (total 33 columns):
 #   Column             Dtype   
---  ------             -----   
 0   fecha_dato         category
 1   ncodpers           int32   
 2   ind_empleado       category
 3   sexo               category
 4   age                float64 
 5   tiprel_1mes        category
 6   nomprov            category
 7   renta              float64 
 8   segmento           category
 9   ind_ahor_fin_ult1  int8    
 10  ind_aval_fin_ult1  int8    
 11  ind_cco_fin_ult1   int8    
 12  ind_cder_fin_ult1  int8    
 13  ind_cno_fin_ult1   int8    
 14  ind_ctju_fin_ult1  int8    
 15  ind_ctma_fin_ult1  int8    
 16  ind_ctop_fin_ult1  int8    
 17  ind_ctpp_fin_ult1  int8    
 18  ind_deco_fin_ult1  int8    
 19  ind_deme_fin_ult1  int8    
 20  ind_dela_fin_ult1  int8    
 21  ind_ecue_fin_ult1  int8    
 22  ind_fond_fin_ult1  int8    
 23  ind_hip_fin_ult1   int8    
 24  ind_plan_fin_ult1  int8    
 25  

## Q1. 마지막 달 도시별 통계 (평균 소득 상위 5개 도시)

In [ ]:
# Q1: 마지막 달 데이터만 추출
last = df[df["fecha_dato"] == "2016-05-28"].copy()
print(last.shape)

g = last.groupby("nomprov", observed=True)

q1 = pd.DataFrame({
    "customers": g["ncodpers"].nunique(),   # 고객 수
    "mean_renta": g["renta"].mean(),        # 평균 소득
    "median_renta": g["renta"].median(),    # 소득 중앙값
    "mean_age": g["age"].mean(),            # 평균 나이
})

# 고객 수 50명 이상인 도시만 남기고, 평균 소득 상위 5개
q1 = q1[q1["customers"] >= 50]
q1.sort_values("mean_renta", ascending=False).head(5)

(931453, 33)


,customers,mean_renta,median_renta,mean_age
nomprov,,,,
CEUTA,513,200515.885152,128699.580,44.007797
MADRID,298841,178362.993637,138024.135,45.205564
"BALEARS, ILLES",9143,171890.951564,122837.220,39.748770
BARCELONA,88754,164584.843157,130877.160,39.865144
MELILLA,641,149071.944149,117718.365,45.329173


In [ ]:
# 검증: 위 Q1 표의 MADRID 값을 groupby 없이 직접 다시 계산해 일치하는지 확인
city = "MADRID"
check = last[last["nomprov"] == city]
print(len(check), check["renta"].mean(), check["renta"].median(), check["age"].mean())

298841 178362.9936366261 138024.135 45.2055641628826


## Q1 결론
2016-05-28 데이터에서 고객 수 50명 이상인 도시 중 평균 소득 상위 5개는 다음과 같다.

1. CEUTA (고객 513명, 평균 소득 200,515.89, 중앙값 128,699.58, 평균 나이 44.0)
2. MADRID (298,841명, 178,362.99, 138,024.14, 45.2)
3. BALEARS, ILLES (9,143명, 171,890.95, 122,837.22, 39.7)
4. BARCELONA (88,754명, 164,584.84, 130,877.16, 39.9)
5. MELILLA (641명, 149,071.94, 117,718.37, 45.3)

- 소득(renta)과 나이의 결측값은 제외하고 평균·중앙값을 계산했다.
- 고객 수는 소득 결측 고객도 포함한 수이다.
- .agg를 쓰지 않고 통계를 각각 계산해 하나의 표로 합쳤다.

## Q2. 전월 대비 신규 가입이 가장 많은 상품 5개

In [ ]:
# Q2: 고객별 · 날짜순으로 정렬
s = df[["ncodpers", "fecha_dato"] + product_cols].copy()
s["month"] = s["fecha_dato"].cat.codes          # 날짜를 0~16 번호로
s = s.sort_values(["ncodpers", "month"]).reset_index(drop=True)

# 윗줄이 "같은 고객의 바로 전달"인 행만 유효
prev_cust = s["ncodpers"].shift(1, fill_value=-1)
prev_month = s["month"].shift(1, fill_value=-1)
valid = (s["ncodpers"] == prev_cust) & (s["month"] - prev_month == 1)

# 전월 0 → 이번 달 1 이면 신규 가입
prev = s[product_cols].shift(1, fill_value=0)
new = (s[product_cols] == 1) & (prev == 0)
new = new[valid]

# 월별 · 상품별 신규 가입 건수
monthly = new.groupby(s.loc[valid, "fecha_dato"], observed=True).sum()
monthly

,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
fecha_dato,,,,,,,,,,,,,,,,,,,,,
2015-02-28,1,1,2132,12,1884,7,291,249,171,348,...,5,27,9,217,3471,259,8,5659,5656,8285
2015-03-28,0,1,2293,15,1786,18,272,284,165,481,...,7,24,12,217,5076,205,4,4479,4473,9135
2015-04-28,0,1,2366,12,1984,10,217,278,167,263,...,4,22,17,871,4627,194,5,2663,5628,8346
2015-05-28,0,0,2264,7,1900,10,205,289,172,238,...,9,21,11,343,4071,201,4,3369,3345,7085
2015-06-28,0,0,6588,9,1857,7,210,213,151,289,...,4,19,8,2931,4655,152,3,5070,8135,9023
2015-07-28,0,0,3056,10,2771,18,191,252,164,265,...,6,27,5,589,4886,232,9,4408,4850,9172
2015-08-28,0,0,5730,10,2412,37,240,231,145,322,...,1,22,13,264,4379,445,2,2950,2968,7115
2015-09-28,0,0,4389,7,1775,22,471,191,122,387,...,8,21,13,271,4463,328,2,4683,4753,11210
2015-10-28,0,0,5682,3,2409,43,468,223,153,349,...,7,19,9,593,4461,911,2,4818,4790,12100


In [ ]:
total = monthly.sum().sort_values(ascending=False)
total.head(5)

ind_recibo_ult1      153146
ind_nom_pens_ult1     84709
ind_nomina_ult1       73753
ind_tjcr_fin_ult1     69118
ind_cco_fin_ult1      69103
dtype: int64

In [ ]:
p = total.index[0]   # 1등 상품
a = df.loc[df["fecha_dato"] == "2016-04-28", ["ncodpers", p]]
b = df.loc[df["fecha_dato"] == "2016-05-28", ["ncodpers", p]]
m = a.merge(b, on="ncodpers", suffixes=("_prev", "_cur"))
print(((m[p + "_prev"] == 0) & (m[p + "_cur"] == 1)).sum())
print(monthly.loc["2016-05-28", p])

10163
10163


## Q2 결론
전체 기간 중 신규 가입(전월 0 → 당월 1)이 가장 많은 상품 5개는 다음과 같다.

1. ind_recibo_ult1 (153,146건)
2. ind_nom_pens_ult1 (84,709건)
3. ind_nomina_ult1 (73,753건)
4. ind_tjcr_fin_ult1 (69,118건)
5. ind_cco_fin_ult1 (69,103건)

- 고객·날짜순으로 정렬한 뒤 shift로 전월 값을 가져와 비교했다(.diff, .agg 미사용).
- 바로 윗줄이 같은 고객의 직전 달인 경우만 집계했다.
  고객의 첫 기록이나 중간에 달이 빠진 경우는 전월 값이 없으므로 제외했다.
- 상품 컬럼의 결측값은 미보유(0)로 처리했다.

## Q3. 상품 3개 이상 보유 여부와 가장 관련성이 큰 기준 (chi-square)

In [ ]:
import numpy as np
from scipy.stats import chi2_contingency

# 고객별 보유 상품 개수 → 3개 이상 여부
last["n_products"] = last[product_cols].sum(axis=1)
last["target"] = np.where(last["n_products"] >= 3, "3+", "<3")
print(last["target"].value_counts())

# 나이 구간: 30 미만 young, 30~64 adult, 65 이상 senior
last["age_group"] = pd.cut(last["age"], bins=[-np.inf, 30, 65, np.inf],
                           right=False, labels=["young", "adult", "senior"])
print(last["age_group"].value_counts())

target
<3    808432
3+    123021
Name: count, dtype: int64
age_group
adult     490900
young     348610
senior     91943
Name: count, dtype: int64


In [ ]:
criteria = {
    "1) sexo": "sexo",
    "2) age_group": "age_group",
    "3) segmento": "segmento",
    "4) ind_empleado": "ind_empleado",
    "5) tiprel_1mes": "tiprel_1mes",
}

rows = []
for name, col in criteria.items():
    ct = pd.crosstab(last[col], last["target"])   # 교차표 (빈 값은 자동 제외)
    ct = ct.loc[ct.sum(axis=1) > 0]               # 고객이 0명인 범주 제거
    print(ct, "\n")

    chi2, p, dof, expected = chi2_contingency(ct)
    n = ct.to_numpy().sum()
    rows.append({"criterion": name, "chi2": chi2, "dof": dof, "p_value": p,
                 "cramers_v": np.sqrt(chi2 / n), "min_expected": expected.min()})

q3 = pd.DataFrame(rows).set_index("criterion").sort_values("chi2", ascending=False)
q3

target     3+      <3
sexo                 
H       46737  379129
V       76284  429298 

target         3+      <3
age_group                
young        8013  340597
adult      102166  388734
senior      12842   79101 

target                 3+      <3
segmento                         
01 - TOP            23757   11658
02 - PARTICULARES   88519  454077
03 - UNIVERSITARIO  10714  335703 

target            3+      <3
ind_empleado                
N             122680  808253
A                130      18
B                115     104
F                 95      57
S                  1       0 

target           3+      <3
tiprel_1mes                
A            121255  271352
I              1739  532690
P                 0     330
R                 1      76 



,chi2,dof,p_value,cramers_v,min_expected
criterion,,,,,
5) tiprel_1mes,183808.591523,3,0.000000e+00,0.445184,10.211533
3) segmento,124374.548799,2,0.000000e+00,0.366800,4711.768629
2) age_group,61002.202414,2,0.000000e+00,0.255913,12143.307073
1) sexo,3412.231367,1,0.000000e+00,0.060526,56246.254419
4) ind_empleado,1343.689088,4,1.120822e-289,0.037981,0.132074


## Q3 결론
가입 상품 3개 이상 여부와 가장 관련성이 큰 기준은 5) tiprel_1mes이다.
chi-square 값(약 183,809)과 Cramér's V(0.445) 모두 다섯 기준 중 가장 크다.
활동 고객(A)의 30.9%가 3개 이상 보유한 반면 비활동 고객(I)은 0.3%에 그쳤다.
ind_empleado는 기대빈도 5 미만인 칸이 있어 해석에 주의가 필요하다.